# Ejercicio 03-01-basic-openai

## Objetivo

Utilizar Google Collab para crear un código Python que conecte con el API del LLM de OpenAI y que nos permite trabajar sobre la respuesta


## Requisitos previos

* Cuenta activa en OpenAI Platform, con créditos y con un API Key generada
* Cuenta de Google Personal o Corporativa
* Configuración de los secretos de la API Keys de OpenAI en Google Collab
  * Se habría realizado en el módulo 00-getting-started, pero si no se ha realizado, se pueden configurar en este ejercicio siguiendo esas instrucciones

# Configuración

## Paso 1: Instalar Dependencias

Pasos a seguir:

* Instalar la SDK de OpenAI
  * https://pypi.org/project/openai/
* Verificar la versión de instalación de la SDK


In [1]:
# Install SDK "OpenAI"
!pip install openai

# Verify installation
import openai

print(f"✅ openai {openai.__version__}")

✅ openai 2.37.0


## Paso 2: Cargar la API Key

Pasos a seguir:

* Acceder a los secretos desde Google Collab
* Identificar el API Key de Acceso de OpenAI
* Mapear el secreto de Google Collab con la variable de entorno de OpenAI


In [2]:
# Prepare imports
import os
from google.colab import userdata

# Load API keys from Colab Secrets for OpenAI
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY_LAB")

print("✅ OpenAI API Key loaded successfully")

✅ OpenAI API Key loaded successfully


## Paso 3: Crear un Cliente de OpenAI

Pasos a seguir:

* Utilizar el constructor de la SDK de OpenAI con los parámetros necesarios

Nota: No se añadirá ningún parámetro extra para este caso

In [3]:
# Prepare imports
from openai import OpenAI

# Create client for OpenAI
openai_client = OpenAI()

print("✅ Client OpenAI configured successfully")

✅ Client OpenAI configured successfully


# Helpers

Se han definido una serie de funciones de soporte para el desarrollo de la parte técnica y los ejemplos utilizados

* Funciones de soporte a metadatos
* Funciones de soporte a costes

## Paso 1: Cargar funciones de ayuda para obtener metainformación de la respuesta en OpenAI

Se puede obtener diferente información sobre lo que ha ocurrido en la respuesta a un LLM: modelo utilizado, tokens de entrada / salida, etc.

**Importante:** Suele depender del tipo de respuesta que facilite el API del proveedor con el que se ha consultado

In [4]:
import json

# ************************
#  Basic
# ************************

def get_response_metadata_basic_openai(response) -> dict:
    return {
        "metadata": {
            "provider": "openai",
            "id": response.id,
            "model": response.model,
            "status": response.status,
            "usage": {
                "input_tokens": response.usage.input_tokens,
                "output_tokens": response.usage.output_tokens,
                "total_tokens": response.usage.total_tokens
            }
        }
    }

# ************************
#  Generic
# ************************

def safe_get(obj, attr, default=None):
    if obj is None:
        return default

    if isinstance(obj, dict):
        return obj.get(attr, default)

    return getattr(obj, attr, default)

def get_response_metadata_openai(response) -> dict:
    usage = safe_get(response, "usage")

    input_tokens = safe_get(usage, "input_tokens", 0) or 0
    output_tokens = safe_get(usage, "output_tokens", 0) or 0
    total_tokens = safe_get(usage, "total_tokens", 0) or 0

    input_tokens_details = safe_get(usage, "input_tokens_details")
    output_tokens_details = safe_get(usage, "output_tokens_details")

    cached_input_tokens = safe_get(
        input_tokens_details,
        "cached_tokens",
        0
    ) or 0

    reasoning_tokens = safe_get(
        output_tokens_details,
        "reasoning_tokens",
        0
    ) or 0

    visible_output_tokens = max(output_tokens - reasoning_tokens, 0)
    non_cached_input_tokens = max(input_tokens - cached_input_tokens, 0)

    return {
        "metadata": {
            "provider": "openai",
            "id": safe_get(response, "id"),
            "model": safe_get(response, "model"),
            "status": safe_get(response, "status"),
            "created_at": safe_get(response, "created_at"),
            "usage": {
                "input_tokens": input_tokens,
                "input_cached_tokens": cached_input_tokens,
                "input_non_cached_tokens": non_cached_input_tokens,
                "output_tokens": output_tokens,
                "output_visible_tokens": visible_output_tokens,
                "output_reasoning_tokens": reasoning_tokens,
                "total_tokens": total_tokens
            }
        }
    }

print("✅ OpenAI response metadata functions loaded")

✅ OpenAI response metadata functions loaded


## Paso 2: Cargar funciones para obtener los costes de la respuesta en OpenAI

Se aconseja acceder a las tarifas del proveedor

**Importante**: Suelen cambiar sin avisar

Recursos:

* OpenAI: https://openai.com/api/pricing


In [5]:
PRICING = {
    # OpenAI (USD per 1M tokens)
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-4o-mini-2024-07-18": {"input": 0.15, "output": 0.60},
    "gpt-5.4-mini": {"input": 0.75, "output": 4.50},
    "gpt-5.4": {"input": 2.50, "output": 15.00},
    "gpt-5.4-nano": {"input": 0.20, "output": 1.25},
    "gpt-5-mini": {"input": 0.25, "output": 2.00},
    "gpt-5-nano": {"input": 0.05, "output": 0.40},
    "gpt-5.5-2026-04-23": {"input": 5.00, "output": 30.00}
}

def get_model_prices(model):
    prices = PRICING.get(model)

    if prices:
        return prices

    for key in PRICING:
        if key in model or model in key:
            return PRICING[key]

    return None

def money(value, decimals=2):
    return f"${value:,.{decimals}f}"

# ************************
#  Basic
# ************************

def get_calculate_cost_basic_openai(model, input_tokens, output_tokens):
    prices = get_model_prices(model)

    if not prices:
        return None

    input_tokens_cost_usd = (input_tokens / 1_000_000) * prices["input"]
    output_tokens_cost_usd = (output_tokens / 1_000_000) * prices["output"]

    total_tokens_cost_usd = input_tokens_cost_usd + output_tokens_cost_usd

    return {
        "cost": {
            "currency_type": "USD",
            "input": round(input_tokens_cost_usd,7),
            "output": round(output_tokens_cost_usd,7),
            "total": round(total_tokens_cost_usd,7)
        }
    }

# ************************
#  Advance
# ************************

def get_calculate_cost_advance_openai(model, input_tokens, output_tokens, reasoning_tokens=0):
    prices = get_model_prices(model)

    if not prices:
        return None

    reasoning_tokens = reasoning_tokens or 0

    visible_output_tokens = max(output_tokens - reasoning_tokens, 0)

    input_tokens_cost_usd = (input_tokens / 1_000_000) * prices["input"]

    visible_output_cost_usd = (
        visible_output_tokens / 1_000_000
    ) * prices["output"]

    reasoning_tokens_cost_usd = (
        reasoning_tokens / 1_000_000
    ) * prices["output"]

    output_tokens_cost_usd = (
        output_tokens / 1_000_000
    ) * prices["output"]

    total_tokens_cost_usd = input_tokens_cost_usd + output_tokens_cost_usd

    return {
        "cost": {
            "currency_type": "USD",
            "input": round(input_tokens_cost_usd, 7),
            "output_visible": round(visible_output_cost_usd, 7),
            "output_reasoning": round(reasoning_tokens_cost_usd, 7),
            "output_total": round(output_tokens_cost_usd, 7),
            "total": round(total_tokens_cost_usd, 7)
        }
    }

# ************************
#  Generic
# ************************

def get_calculate_cost_openai(response):
    model = response.model
    usage = response.usage

    prices = get_model_prices(model)

    if not prices:
        return None

    input_tokens = usage.input_tokens
    output_tokens = usage.output_tokens

    output_tokens_details = safe_get(usage, "output_tokens_details")
    reasoning_tokens = safe_get(output_tokens_details, "reasoning_tokens", 0) or 0

    visible_output_tokens = max(output_tokens - reasoning_tokens, 0)

    input_tokens_cost_usd = (input_tokens / 1_000_000) * prices["input"]
    output_tokens_cost_usd = (output_tokens / 1_000_000) * prices["output"]

    reasoning_tokens_cost_usd = (reasoning_tokens / 1_000_000) * prices["output"]
    visible_output_tokens_cost_usd = (visible_output_tokens / 1_000_000) * prices["output"]

    total_tokens_cost_usd = input_tokens_cost_usd + output_tokens_cost_usd

    return {
        "cost": {
            "currency_type": "USD",
            "model": model,
            "input": round(input_tokens_cost_usd, 7),
            "output": round(output_tokens_cost_usd, 7),
            "output_visible": round(visible_output_tokens_cost_usd, 7),
            "output_reasoning": round(reasoning_tokens_cost_usd, 7),
            "total": round(total_tokens_cost_usd, 7)
        }
    }


print("✅ Cost calculation functions loaded")

✅ Cost calculation functions loaded


# Invocación de una llamada básica


## Paso 1: Llamada básica + Respuesta

Genera una respuesta general que NO esta sujeta a ningún tipo de parámetro

NO se utilizará ningún "system prompt"

**Importante**: Esto afecta al coste y a la calidad

Recursos de la respuesta:

* https://developers.openai.com/api/docs/guides/migrate-to-responses

In [ ]:
# Define user prompt
USER_PROMPT = "What should I consider when modernizing an application?"

print("***** Llamada básica *****")

# Invoke LLM
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=USER_PROMPT
)

# Get text response
#response_text = response.output[0].content[0].text
response_text = response.output_text

# Show response
print("Respuesta del modelo:\n")
print(response_text)

***** Llamada básica *****
Respuesta del modelo:

Modernizing an application can be a complex process, and there are several factors to consider to ensure a successful transition. Here’s a comprehensive list:

### 1. **Assessment of Current State**
   - **Technology Stack**: Evaluate the existing technologies, frameworks, and languages.
   - **Architecture**: Understand the current architecture (monolithic, microservices, etc.).
   - **Dependencies**: Identify libraries, services, and other dependencies that may need updates.

### 2. **Business Goals**
   - **Objectives**: Determine the business objectives for modernization (e.g., performance improvement, cost reduction).
   - **User Needs**: Gather user feedback to understand their needs and pain points.

### 3. **Target Architecture**
   - **Cloud vs. On-Premises**: Decide whether to move to the cloud, stay on-premises, or adopt a hybrid model.
   - **Microservices vs. Monolithic**: Consider breaking down monolithic applications into

## Paso 2: Obtener metadatos

### Modo 1: Mostrar metadatos básicos

In [ ]:
# Get metadata basic response
metadata_json = get_response_metadata_basic_openai(response)

# Show metadata
print("***** Basic Metadata *****")
print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

***** Basic Metadata *****
{
  "metadata": {
    "id": "resp_0076f983ee70fc0b006a1715ee5a9081a193954a9c7c01e604",
    "model": "gpt-4o-mini-2024-07-18",
    "status": "completed",
    "usage": {
      "input_tokens": 17,
      "output_tokens": 765,
      "total_tokens": 782
    }
  }
}


### Modo 2: Mostrar metadatos

In [ ]:
# Get metadata generic response
metadata_json = get_response_metadata_openai(response)

# Show metadata
print("***** Generic Metadata *****")
print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

***** Generic Metadata *****
{
  "metadata": {
    "id": "resp_0076f983ee70fc0b006a1715ee5a9081a193954a9c7c01e604",
    "model": "gpt-4o-mini-2024-07-18",
    "status": "completed",
    "created_at": 1779897838.0,
    "usage": {
      "input_tokens": 17,
      "input_cached_tokens": 0,
      "input_non_cached_tokens": 17,
      "output_tokens": 765,
      "output_visible_tokens": 765,
      "output_reasoning_tokens": 0,
      "total_tokens": 782
    }
  }
}


## Paso 3: Obtener costes

### Modo 1: Cálculo básico SIN función de costes específica

In [ ]:
# Pricing and Conditions
# - Estimated pricing for gpt-4.1-mini (USD per 1M tokens)
# - Cached tokens are not included
# - Adjust these values ​​according to the current official pricing
INPUT_PRICE_PER_MILLION = 0.15
OUTPUT_PRICE_PER_MILLION = 0.60

# Cost calculation
input_cost = (response.usage.input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
output_cost = (response.usage.output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
total_cost = input_cost + output_cost

# Show cost response
print("\n***** Manual Cost *****")
print(f"Input cost:  ${input_cost:.7f}")
print(f"Output cost: ${output_cost:.7f}")
print(f"Total cost:  ${total_cost:.7f}")


***** Manual Cost *****
Input cost:  $0.0000025
Output cost: $0.0004590
Total cost:  $0.0004615


### Modo 2: Cálculo básico CON función de costes específica

In [ ]:
# Get basic cost response
cost_json = get_calculate_cost_basic_openai(
    model=response.model,
    input_tokens=response.usage.input_tokens,
    output_tokens=response.usage.output_tokens
)

# Show cost response
print("***** Basic Cost *****")
print(json.dumps(cost_json, indent=2, ensure_ascii=False))

***** Basic Cost *****
{
  "cost": {
    "currency_type": "USD",
    "input": 2.5e-06,
    "output": 0.000459,
    "total": 0.0004615
  }
}


### Modo 3: Cálculo avanzado CON función de costes específica

In [ ]:
# Get advance cost response
cost_json = get_calculate_cost_advance_openai(
    model=response.model,
    input_tokens=response.usage.input_tokens,
    output_tokens=response.usage.output_tokens
)

# Show cost response
print("***** Advance Cost *****")
print(json.dumps(cost_json, indent=2, ensure_ascii=False))

***** Advance Cost *****
{
  "cost": {
    "currency_type": "USD",
    "input": 2.5e-06,
    "output_visible": 0.000459,
    "output_reasoning": 0.0,
    "output_total": 0.000459,
    "total": 0.0004615
  }
}


### Modo 4: Cálculo avanzado CON función de costes genérica

In [ ]:
# Get generic cost response
cost_json = get_calculate_cost_openai(response)

# Show cost response
print("***** Generic Cost *****")
print(json.dumps(cost_json, indent=2, ensure_ascii=False))

***** Generic Cost *****
{
  "cost": {
    "currency_type": "USD",
    "model": "gpt-4o-mini-2024-07-18",
    "input": 2.5e-06,
    "output": 0.000459,
    "output_visible": 0.000459,
    "output_reasoning": 0.0,
    "total": 0.0004615
  }
}


# System prompt

El "system prompt" facilita disponer de un contrato de calidad

Este prompt suele ir evolucionando con el tiempo



## Consideración 1: Uso de system prompt básico con Rol

La respuesta puede estar un poco más enfocada y ser un poco mejor a la invocación que solamente se utiliza el prompt del usuario (terminología más técnica, respuesta más acotada, etc)

Nota: Puede que NO haya mucha diferencia con el anterior caso



In [6]:
# Define system prompt: Role
SYSTEM_PROMPT = """
Acts as a senior software architect with experience in application modernization
"""

# Define User Prompt
USER_PROMPT = "What should I consider when modernizing an application?"

print("***** SYSTEM PROMPT - Consideración 1: Rol básico *****")

# Invoke LLM
response = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions=SYSTEM_PROMPT,
    input=USER_PROMPT
)

# Get text response
#response_text = response.output[0].content[0].text
response_text = response.output_text

# Show
print("Respuesta del modelo:\n")
print(response_text)

# Get metadata response
metadata_json = get_response_metadata_openai(response)

# Show metadata
print("\n***** Metadata *****")
print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

# Get cost response
cost_json = get_calculate_cost_openai(response)

# Show metadata
print("\n***** Cost *****")
print(json.dumps(cost_json, indent=2, ensure_ascii=False))

***** SYSTEM PROMPT - Consideración 1: Rol básico *****
Respuesta del modelo:

Modernizing an application involves several critical considerations to ensure that the transition achieves the desired outcomes while minimizing risks. Here are key factors to consider:

### 1. **Business Objectives**
   - **Alignment with Goals**: Ensure that the modernization aligns with business strategy and objectives.
   - **User Needs**: Understand how users interact with the application and what improvements they need.

### 2. **Current Architecture**
   - **Assess Existing Functionality**: Identify which features are critical and which can be deprecated.
   - **Technical Debt**: Evaluate the technical debt that could hinder modernization efforts.

### 3. **Technology Stack**
   - **Choosing Modern Technologies**: Evaluate which modern frameworks, databases, and cloud services would provide flexibility and scalability.
   - **Interoperability**: Ensure that new technologies can integrate with existing

## Consideración 2: Uso de system prompt con Rol + Audiencia + Formato de respuesta

Con esta opción ya si que se pueden poner parámetros que pueden afectar al coste

Ejemplo: Poner ejemplos de cada cosa (incrementa los tokens), limitar el formato de la respuesta (palabras, etc.)

El parámetro que más condiciona la respuesta es el formato de la respuesta ya que en la mayoría de los casos son restricciones

In [ ]:
# Define system prompt: Role + Audience + Format
SYSTEM_PROMPT = """
Acts as a senior software architect with experience in application modernization

The audience will be a team of backend developers with 5+ years of experience.

The response format will be:
- Write in prose
- Do not use tables, emojis, bullet points o numbered list
- Maximum of 300 words
"""

# Define User Prompt
USER_PROMPT = "What should I consider when modernizing an application?"

print("***** SYSTEM PROMPT - Consideración 2: Rol + Audiencia + Formato *****")

# Invoke LLM
response = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions=SYSTEM_PROMPT,
    input=USER_PROMPT
)

# Get text response
#response_text = response.output[0].content[0].text
response_text = response.output_text

# Show
print("Respuesta del modelo:\n")
print(response_text)

# Get metadata response
metadata_json = get_response_metadata_openai(response)

# Show metadata
print("\n***** Metadata *****")
print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

# Get cost response
cost_json = get_calculate_cost_openai(response)

# Show cost response
print("\n***** Cost *****")
print(json.dumps(cost_json, indent=2, ensure_ascii=False))

***** SYSTEM PROMPT - Consideración 2: Rol + Audiencia + Formato *****
Respuesta del modelo:

When embarking on the modernization of an application, several key factors need careful consideration to ensure a successful transition. First and foremost, you must understand the current application architecture. A thorough assessment of its components, dependencies, and integration points will reveal potential roadblocks and opportunities for improvement. Engaging with stakeholders to clarify the primary goals of modernization—whether it’s enhancing performance, scalability, or security—will guide your decision-making process.

You are likely to confront various architectural styles, such as microservices or serverless, which may better suit your needs compared to the existing monolithic structure. Carefully evaluating how these paradigms can decompose functionalities into manageable services is crucial. Additionally, consider leveraging containerization and orchestration tools to facilitat

## Consideración 3: Uso de system prompt con Rol + Contexto + Audiencia + Formato de respuesta

Con esta opción podemos mejorar la calidad de la respuesta

Ejemplo: Poner ejemplos de cada cosa (incrementa los tokens), limitar el formato de la respuesta (palabras, etc.)

El "parámetro" que más condiciona la respuesta es el formato de la respuesta ya que en la mayoría de los casos son restricciones

In [ ]:
# Define system prompt: Role + Context + Tone + Audience + Format
SYSTEM_PROMPT = """
Act as a senior software architect with 15+ years of experience in application modernization.

Your goal is to help the user analyze, design, and propose practical technical solutions to modernize legacy or existing applications.

Remember to keep best practices in mind.

The audience will be a team of backend developers with 5+ years of experience.

The communication style will be didactic and professional, using clear explanations, a practical approach, and actionable recommendations.

Response rules:
- Write in prose
- Do not use tables, emojis, bullet points o numbered list
- Maximum of 300 words
"""

# Define User Prompt
USER_PROMPT = "What should I consider when modernizing an application?"

print("***** SYSTEM PROMPT - Consideración 3: Rol + Contexto + Tono + Audiencia + Formato *****")

# Invoke LLM
response = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions=SYSTEM_PROMPT,
    input=USER_PROMPT
)

# Get text response
#response_text = response.output[0].content[0].text
response_text = response.output_text

# Show
print("Respuesta del modelo:\n")
print(response_text)

# Get metadata response
metadata_json = get_response_metadata_openai(response)

# Show metadata
print("\n***** Metadata *****")
print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

# Get cost response
cost_json = get_calculate_cost_openai(response)

# Show cost response
print("\n***** Cost *****")
print(json.dumps(cost_json, indent=2, ensure_ascii=False))

***** SYSTEM PROMPT - Consideración 3: Rol + Contexto + Tono + Audiencia + Formato *****
Respuesta del modelo:

When modernizing an application, several critical considerations can guide your approach to ensure a successful transition while minimizing risks and maximizing benefits.

Firstly, you need to understand the core functionality and architecture of the existing application. Conduct a thorough assessment to identify the application’s strengths, weaknesses, and performance bottlenecks. This understanding will inform decisions on whether to refactor, rearchitect, or replace components.

Next, define clear modernization objectives aligned with business goals. This may include improving scalability, enhancing user experience, or integrating with modern ecosystems. Clarity in objectives ensures that all team members are aligned on the vision.

Choosing the right technology stack is another crucial aspect. Evaluate contemporary frameworks and tools that best suit your application's re

## Consideración 4: Uso de system prompt con: Rol + Contexto + Audiencia + Formato de respuesta + Restricciones de comportamiento + Gestión de Casos Límite

Con esta opción podemos mejorar la calidad de la respuesta

Ejemplo: Poner ejemplos de cada cosa (incrementa los tokens), limitar el formato de la respuesta (palabras, etc.)

El "parámetro" que más condiciona la respuesta es el formato de la respuesta ya que en la mayoría de los casos son restricciones

In [ ]:
# Define system prompt: Role + Context + Tone + Audience + Format + Behavioral constraints + Edge case handling
SYSTEM_PROMPT = """
Act as a senior software architect with 15+ years of experience in application modernization.

Your goal is to help the user analyze, design, and propose practical technical solutions to modernize legacy or existing applications.

Remember to keep best practices in mind.

The audience will be a team of backend developers with 5+ years of experience.

The communication style will be didactic and professional, using clear explanations, a practical approach, and actionable recommendations.

Response rules:
- Write in prose
- Do not use tables, emojis, bullet points o numbered list
- Maximum of 300 words
- Prioritize actionable insights over generic theory
- Always mention at least one non-obvious risk that teams commonly overlook
- Do not recommend fashionable patterns such as microservices, event-driven architecture, Kubernetes or cloud migration unless they are justified by the application’s constraints, team maturity, operational model and business goals.
- If the user provides insufficient context about the legacy system, explicitly state the assumptions being made and propose a safe first step, such as assessment, observability, dependency mapping or incremental refactoring, instead of giving a definitive modernization plan.
"""

# Define User Prompt
USER_PROMPT = "What should I consider when modernizing an application?"

print("***** SYSTEM PROMPT - Consideración 4: Rol + Contexto + Tono + Audiencia + Formato + Restricciones de comportamiento + Gestión de Casos Límite *****")

# Invoke LLM
response = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions=SYSTEM_PROMPT,
    input=USER_PROMPT
)

# Get text response
#response_text = response.output[0].content[0].text
response_text = response.output_text

# Show
print("Respuesta del modelo:\n")
print(response_text)

# Get metadata response
metadata_json = get_response_metadata_openai(response)

# Show metadata
print("\n***** Metadata *****")
print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

# Get cost response
cost_json = get_calculate_cost_openai(response)

# Show cost response
print("\n***** Cost *****")
print(json.dumps(cost_json, indent=2, ensure_ascii=False))

***** SYSTEM PROMPT - Consideración 4: Rol + Contexto + Tono + Audiencia + Formato + Restricciones de comportamiento + Gestión de Casos Límite *****
Respuesta del modelo:

When modernizing an application, it is crucial to first conduct a thorough assessment of the existing system. This involves evaluating its architecture, dependencies, performance metrics, and business alignment. Understanding how the application fits within the broader system landscape can reveal critical insights, including potential bottlenecks and outdated technologies.

One of the key actions to undertake is to establish observability within the current system. Implementing logging, monitoring, and tracing tools can provide deep insights into performance issues and allow your team to make informed decisions during the modernization process. This approach avoids the common pitfall of failing to address hidden flaws or inefficiencies that may otherwise amplify during modernization efforts.

It is also important to 

## Consideración 5: Uso de system prompt con contexto avanzado estructurado



In [ ]:
# Define system prompt with structure (Production-grade prompt)
PRODUCTION_SYSTEM_PROMPT = """
Act as a senior software architect with 15+ years of experience in application modernization.

# Objective
Your goal is to help the user analyze, design, and propose practical technical solutions to modernize legacy or existing applications, always considering software engineering best practices.

# Tone
- Didactic and professional
- Clear and structured explanations
- Practical and action-oriented
- Technically rigorous, but easy for experienced backend developers to follow

# Audience
- Backend developers with 5+ years of experience
- Technically proficient professionals familiar with software architecture, APIs, databases, deployment, and operational concerns

# Output Format
- Write in prose
- Do not use tables, emojis, bullet points o numbered list
- Maximum of 300 words
- Concise and actionable content
- Avoid generic statements; provide concrete technical insights

# Behavior Guidelines
- Prioritize clarity, precision, and real-world applicability
- Recommend solutions aligned with maintainability, scalability, security, observability, and operational simplicity
- Explain trade-offs when relevant
- Focus on practical modernization paths such as refactoring, modularization, API design, cloud readiness, CI/CD, testing, observability, and incremental migration
- Avoid over-explaining basic concepts unless necessary
- If key context is missing, make reasonable assumptions or ask concise, targeted questions
- Never repeat informaticion the user already provided

Do not:
- Use filler phrases like 'Great question!' or 'There are many things to consider'
- Repeat information the user already provided
"""

# Define User Prompt
USER_PROMPT = "What should I consider when modernizing an application?"

print("***** SYSTEM PROMPT - Consideración 5: Production-grade prompt *****")

# Invoke LLM
response = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions=PRODUCTION_SYSTEM_PROMPT,
    input=USER_PROMPT
)

# Get text response
#response_text = response.output[0].content[0].text
response_text = response.output_text

# Show
print("Respuesta del modelo:\n")
print(response_text)

# Get metadata response
metadata_json = get_response_metadata_openai(response)

# Show metadata
print("\n***** Metadata *****")
print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

# Get cost response
cost_json = get_calculate_cost_openai(response)

# Show cost response
print("\n***** Cost *****")
print(json.dumps(cost_json, indent=2, ensure_ascii=False))

***** SYSTEM PROMPT - Consideración 5: Production-grade prompt *****
Respuesta del modelo:

When modernizing an application, focus on several key aspects. First, assess the architecture of the existing system. Identify components that can be modularized to enhance maintainability and scalability. A microservices architecture can be beneficial, allowing for independent deployment and development cycles. 

Next, evaluate the data layer. Consider transitioning to more flexible database solutions, such as NoSQL for unstructured data, or employing sharding and replication strategies for performance. Ensure that the data access is abstracted through well-defined APIs, allowing for smoother interactions and integrations.

Security is paramount; modernize by implementing OAuth, API Gateway patterns, and robust authentication mechanisms. Enhance observability by incorporating logging and monitoring tools that provide real-time insights, enabling proactive issue resolution.

Define a clear CI/CD

# Parámetros del modelo

Existen una serie de características que permiten controlar el comportamiento' del modelo

Cada proveedor tiene sus propios parámetros aunque en muchos casos son compartidos


**IMPORTANTE**
* Los modelos de razonamiento NO suelen aceptar la mayoría de estos parámetros
* En muchos casos estos parámetros esta fijados internamento

## Consideración 1: Parámetro de temperatura

Determina la cantidad de aleatoriedad a la hora de elegir el siguiente token

Suele usarse para añadir determinismo o bien creatividad

En OpenAI se suele trabajar en un rango de: 0.0 a 2.0
En Anthropic se suele trabajar en un rango de: 0.0 a 1.0

Detalle:

* **0.0**: el modelo siempre elige el token más probable. Respuestas casi deterministas
* **0.3 - 0.5**: ligera variación. Bueno para análisis técnico, clasificación, extracción de datos
* **0.7 - 1.0** respuestas más variadas y creativas. Bueno para brainstorming, redacción creativa
* **> 1.0** — alta aleatoriedad. Puede producir resultados incoherentes

**TRUCOS**
* Se aconseja empezar con temperaturas de 0.2-0.3 en producción
* Ir subiendo poco a poco cuando se necesita "creatividad"


### Aplicación del parámetro "temperatura"

In [7]:
# Define system prompt: Role
SYSTEM_PROMPT = """
Acts as a senior software architect with experience in application modernization
"""

# Define User Prompt
USER_PROMPT = "What should I consider when modernizing an application?"

# Configure temperature (test with: 0.0, 0.5, 1.0, 1.5 y 2.0)
temperature = 0.0

print("***** PARAM - Consideración 1: Ejemplo de uso del parámetro temperatura *****")
print(f"Temperatura: {temperature}")

# Invoke LLM
response = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions=SYSTEM_PROMPT,
    input=USER_PROMPT,
    temperature=temperature
)

# Get text response
#response_text = response.output[0].content[0].text
response_text = response.output_text

# Show
print("Respuesta del modelo:\n")
print(response_text)

# Get metadata response
metadata_json = get_response_metadata_openai(response)

# Show metadata
print("\n***** Metadata *****")
print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

# Get cost response
cost_json = get_calculate_cost_openai(response)

# Show cost response
print("\n***** Cost *****")
print(json.dumps(cost_json, indent=2, ensure_ascii=False))



***** PARAM - Consideración 1: Ejemplo de uso del parámetro temperatura *****
Temperatura: 0.0
Respuesta del modelo:

When modernizing an application, there are several key considerations to ensure a successful transition. Here’s a comprehensive list:

### 1. **Assessment of Current State**
   - **Architecture Review**: Analyze the existing architecture to identify bottlenecks, dependencies, and areas for improvement.
   - **Code Quality**: Evaluate the codebase for maintainability, readability, and adherence to best practices.
   - **Technical Debt**: Identify and prioritize technical debt that needs to be addressed during modernization.

### 2. **Business Objectives**
   - **Alignment with Business Goals**: Ensure that the modernization aligns with the overall business strategy and objectives.
   - **User Needs**: Gather feedback from users to understand their needs and pain points.

### 3. **Modernization Strategy**
   - **Lift and Shift vs. Re-architecting**: Decide whether to simp

### Comparativa de costes según diferentes combinaciones temperatura

In [8]:
# Define system prompt: Role
SYSTEM_PROMPT = """
Acts as a senior software architect with experience in application modernization
"""

# Define User Prompt
USER_PROMPT = "What should I consider when modernizing an application?"

# Configure temperature (test with: 0.0, 0.5, 1.0, 1.5 y 2.0)
temperatures = [0.0, 0.7, 1.5]

# Configura num excecutions
num_executions = 3

print("***** PARAM - Consideración 1: Comparativa de costes según la temperatura *****\n")

# Create table header
print(f"{'Temperature':>20s} {'Execution':>12s} {'Input tok':>12s} {'Output tok':>12s} {'Cost':>12s}")
print("-" * 80)

for temperature in temperatures:

    for i in range(num_executions):

      # Invoke LLM
      response = openai_client.responses.create(
          model="gpt-4o-mini",
          instructions=SYSTEM_PROMPT,
          input=USER_PROMPT,
          temperature=temperature
      )

      # Get cost response
      cost_json = get_calculate_cost_openai(response)

      cost_str = f"${cost_json['cost']['total']:.6f}" if cost_json else "N/A"

      # Create table row
      print(f"{str(temperature):>20s} {str((i+1)):>12s} {response.usage.input_tokens:>12d} {response.usage.output_tokens:>12d} {cost_str:>12s}")


***** PARAM - Consideración 1: Comparativa de costes según la temperatura *****

         Temperature    Execution    Input tok   Output tok         Cost
--------------------------------------------------------------------------------
                 0.0            1           34          762    $0.000462
                 0.0            2           34          789    $0.000478
                 0.0            3           34          843    $0.000511
                 0.7            1           34          744    $0.000452
                 0.7            2           34          756    $0.000459
                 0.7            3           34          913    $0.000553
                 1.5            1        10278         4251    $0.004092
                 1.5            2           34         4610    $0.002771
                 1.5            3           34          902    $0.000546


## Consideración 2: Consistencia de respuestas

La variabilidad de las respuestas puede ser muy alta debido principalmente a que los modelos son **estocásticos** (contiene un poco de azar o incertidumbre).

Por otro lado, hay que en cuenta la combinación: TEMPERATURA + INSTRUCCIONES_PRECISAS (USER_PROMPT)

Situaciones

* Temperatura elevada + Instrucción imprecisa = respuestas muy inconsistentes
* Temperatura baja + Instrucción precisa = respuestas consistentes y predecibles

NOTA: Este apartado NO tiene ejecicios/ejemplos asociados

## Consideración 3: Parámetro de máximo número de tokens

Determina la cantidad máxima de tokens que el modelo puede generar.

No se considera la longitud objetivo, se trata del límite superior.

Cuando se alcance este límite la respuesta se corta precipitadamente.

Se suele utilizar para controlar costes de los tokens de salida
Evita que el modelo genera respuestas excesivamente largas

**TRUCOS**
* Es un límite de seguridad -> NO un control de longitud
* Se combian con restricciones de formato

**IMPORTANTE**
* En OpenAI y Gemini es opcional
* En Anthropic es obligatorio (se usa max_tokens)
* Se conseja controlar el estado de salida de la petición ("status")

#### Aplicación del parámetro "máximo número de tokens"

In [ ]:
# Define system prompt: Role
SYSTEM_PROMPT = """
Acts as a senior software architect with experience in application modernization
"""

# Define User Prompt
USER_PROMPT = "What should I consider when modernizing an application?"

# Configure limits
limits = [50, 200, 800]

print("***** PARAM - Consideración 3: Ejemplo de uso del parámetro max_output_tokens *****")

for limit in limits:

    print(f"\n ************************ ")
    print(f"\n [*] Limit = {limit} ")
    print(f"\n ************************ ")

    # Invoke LLM
    response = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions=SYSTEM_PROMPT,
        input=USER_PROMPT,
        temperature=0.3,
        max_output_tokens=limit
    )

    # Check status
    status = response.status
    truncated = " [TRUNCATED]" if status == "incomplete" else " [COMPLETE]"

    print(f"\nmax_output_tokens = {limit}{truncated} *****")
    print(f"Tokens generados: {response.usage.output_tokens}")
    print(f"Respuesta:\n{response.output_text}")

    # Get metadata response
    metadata_json = get_response_metadata_openai(response)

    # Show metadata
    print("\n***** Metadata *****")
    print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

    # Get cost response
    cost_json = get_calculate_cost_openai(response)

    # Show cost response
    print("\n***** Cost *****")
    print(json.dumps(cost_json, indent=2, ensure_ascii=False))

***** PARAM - Consideración 3: Ejemplo de uso del parámetro max_output_tokens *****

 ************************ 

 [*] Limit = 50 

 ************************ 

max_output_tokens = 50 [TRUNCATED] *****
Tokens generados: 50
Respuesta:
Modernizing an application is a complex process that requires careful planning and execution. Here are key considerations to keep in mind:

### 1. **Assessment of Current State**
   - **Architecture Review**: Evaluate the existing architecture (monolithic vs.

***** Metadata *****
{
  "metadata": {
    "id": "resp_0e53b23124a77948006a171b742a48819c83a0168e39a9bd07",
    "model": "gpt-4o-mini-2024-07-18",
    "status": "incomplete",
    "created_at": 1779899252.0,
    "usage": {
      "input_tokens": 34,
      "input_cached_tokens": 0,
      "input_non_cached_tokens": 34,
      "output_tokens": 50,
      "output_visible_tokens": 50,
      "output_reasoning_tokens": 0,
      "total_tokens": 84
    }
  }
}

***** Cost *****
{
  "cost": {
    "currency_type": "U

#### Comparativa de costes según diferentes combinaciones de inputs + outputs

La proporción (normalemente asimétrica) de tokens de entrada y salida suelen condicionar enormemente la factura

In [ ]:
# Define User Prompt
USER_PROMPT = "What should I consider when modernizing an application?"

# Configure limits (max_output_tokens)
token_output_limits = [50, 200, 800]

print("***** PARAM - Consideración 3:  Comparativa de costes según el max_output_tokens *****\n")

# Create table header
print(f"{'max_output_tokens':>20s} {'Input tok':>12s} {'Output tok':>12s} {'Cost':>12s}")
print("-" * 60)

for limit in token_output_limits:

    # Invoke LLM
    response = openai_client.responses.create(
        model="gpt-4o-mini",
        input=USER_PROMPT,
        max_output_tokens=limit,
        temperature=0.3
    )

    # Get cost response
    cost_json = get_calculate_cost_openai(response)

    cost_str = f"${cost_json['cost']['total']:.6f}" if cost_json else "N/A"

    # Create table row
    print(f"{limit:>20d} {response.usage.input_tokens:>12d} {response.usage.output_tokens:>12d} {cost_str:>12s}")

***** PARAM - Consideración 3:  Comparativa de costes según el max_output_tokens *****

   max_output_tokens    Input tok   Output tok         Cost
------------------------------------------------------------
                  50           17           50    $0.000032
                 200           17          200    $0.000122
                 800           17          770    $0.000465


## Consideración 4: Parámetro de razonamiento

**IMPORTANTE**
* Se utilizará el modelo "gpt-5.4" que es un modelo con razonamiento

#### Aplicación del parámetro "reasoning"

In [ ]:
# Define system prompt: Role
SYSTEM_PROMPT = """
Acts as a senior software architect with experience in application modernization
"""

# Define User Prompt
USER_PROMPT = "What should I consider when modernizing an application?"

# Configure reasoning (test with: "low", "medium", "high" y "xhigh")
reasoning_effort = "medium"

print("***** PARAM - Consideración 4: Ejemplo de uso del parámetro reasoning *****")
print(f"Reasoning: {reasoning_effort}")

# Invoke LLM
response = openai_client.responses.create(
    model="gpt-5.4",
    instructions=SYSTEM_PROMPT,
    input=USER_PROMPT,
    reasoning={
        "effort": reasoning_effort
    },
    max_output_tokens=800
)

# Get text response
#response_text = response.output[0].content[0].text
response_text = response.output_text

# Show

print(f"Tokens de razonamiento generados: {response.usage.output_tokens}")
print("Respuesta del modelo:\n")
print(response_text)

# Get metadata response
metadata_json = get_response_metadata_openai(response)

# Show metadata
print("\n***** Metadata *****")
print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

# Get cost response
cost_json = get_calculate_cost_openai(response)

# Show cost response
print("\n***** Cost *****")
print(json.dumps(cost_json, indent=2, ensure_ascii=False))



***** PARAM - Consideración 4: Ejemplo de uso del parámetro reasoning *****
Reasoning: medium
Tokens de razonamiento generados: 800
Respuesta del modelo:

When modernizing an application, the biggest mistake is treating it as only a technology upgrade. Successful modernization is usually a mix of **business, architecture, operations, security, and organizational change**.

Here are the main things to consider.

---

## 1. Start with the business outcome
Before choosing any technology, be clear on **why** you’re modernizing.

Common drivers:
- Reduce operational cost
- Improve scalability and performance
- Increase delivery speed
- Improve reliability
- Address security/compliance gaps
- Replace unsupported tech
- Enable new products/channels/APIs
- Improve developer productivity

Ask:
- What problem are we trying to solve?
- How will we measure success?
- What’s the timeline and urgency?
- What’s the expected ROI?

If the goal is unclear, modernization often becomes an expensive rewrit


# Uso de diferentes modelos


## Consideración 1: Modelos diferentes dentro de un mismo proveedor

In [ ]:
# Define system prompt: Role
SYSTEM_PROMPT = """
Acts as a senior software architect with experience in application modernization
"""

# Define User Prompt
USER_PROMPT = "What should I consider when modernizing an application?"

# Configure OpenAI Models
openai_models = [
    "gpt-4o-mini",
    "gpt-5.4-mini",   # Uncomment if you have access
    "gpt-5.4",        # Uncomment if you have access (more expensive)
]

print("***** DIFERENTES MODELOS - Consideración 1: Modelos diferentes de un mismo proveedor *****")

for model_name in openai_models:

    print(f"\n ************************ ")
    print(f" [*] model = {model_name} ")
    print(f" ************************ ")

    try:

        # Invoke LLM
        response = openai_client.responses.create(
            model=model_name,
            instructions=SYSTEM_PROMPT,
            input=USER_PROMPT,
            temperature=0.3
        )

        print(f"\n{'*' * 60}")

        # Metadata details
        print(f"Modelo: {response.model}")
        print(f"Tokens: {response.usage.input_tokens} input + {response.usage.output_tokens} output = {response.usage.input_tokens + response.usage.output_tokens} tokens")

        # Status details
        status = response.status
        info_status = " [TRUNCATED]" if status == "incomplete" else " [COMPLETE]"

        print(f"Estado Respuesta: {info_status}")

        # Get cost response
        cost_json = get_calculate_cost_openai(response)

        # Cost details
        if cost_json:
            print(f"Coste: ${cost_json['cost']['total']:.6f}")

        # Get metadata response
        metadata_json = get_response_metadata_openai(response)

        # Show metadata
        print("\n***** Metadata *****")
        print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

        # Show cost response
        print("\n***** Cost *****")
        print(json.dumps(cost_json, indent=2, ensure_ascii=False))

        # Response details
        print(f"\nRespuesta del modelo:\n{response.output_text}")
    except Exception as e:
        print(f"\n{'*' * 60}")
        print(f"Model {model_name}: Error — {e}")

***** DIFERENTES MODELOS - Consideración 1: Modelos diferentes de un mismo proveedor *****

 ************************ 
 [*] model = gpt-4o-mini 
 ************************ 

************************************************************
Modelo: gpt-4o-mini-2024-07-18
Tokens: 34 input + 778 output = 812 tokens
Estado Respuesta:  [COMPLETE]
Coste: $0.000472

***** Metadata *****
{
  "metadata": {
    "id": "resp_09309ef702ddd028006a171c227cb08190a772e9ae1cd4d9a6",
    "model": "gpt-4o-mini-2024-07-18",
    "status": "completed",
    "created_at": 1779899426.0,
    "usage": {
      "input_tokens": 34,
      "input_cached_tokens": 0,
      "input_non_cached_tokens": 34,
      "output_tokens": 778,
      "output_visible_tokens": 778,
      "output_reasoning_tokens": 0,
      "total_tokens": 812
    }
  }
}

***** Cost *****
{
  "cost": {
    "currency_type": "USD",
    "model": "gpt-4o-mini-2024-07-18",
    "input": 5.1e-06,
    "output": 0.0004668,
    "output_visible": 0.0004668,
    "output

## Consideración 2: Modelos "equivalentes" en diferentes proveedores con las mismas condiciones de configuración

NOTA: Este apartado NO tiene ejecicios/ejemplos asociados

# Uso de herramientas

Resuelven ciertas situaciones o limitaciones que tienen los modelos

* Conocer cosas en fechas posteriores al final de su entrenamiento
* Acceso a información privada
* Ejecución de código

Son funcionalidades que el proveedor ejecuta en sus propios servidores: el modelo decide cuándo usarlas, las invoca, procesa los resultados y genera la respuesta final.

Hay de muchos tipos: búsqueda en internet, búsqueda en tus ficheros, ejecutar código, generación de imagenes, conectar son MCPs, etc.

**IMPORTANTE**
* NO todos los modelos los soportan
* Algunos de los tipos puede tener un coste adicional sobre el coste de los tokens normales (Revisar la actualización)

## Consideración 1: Uso de la herramienta "web_search"

Facilita al modelo a acceder a internet para buscar informacion que no tiene identificada en su entrenamiento

La característica tools=[{"type": "web_search"}] pone la herramienta a disposición del modelo, pero el modelo puede decidir si la usa o no según la consulta. La documentación indica explícitamente que, en Responses API, el modelo puede elegir buscar o no buscar en función del prompt

In [ ]:
# Define system prompt: Role
SYSTEM_PROMPT = """
Acts as a senior software architect with experience in application modernization
"""

# Define User Prompt
WEB_USER_PROMPT = "What should I consider when modernizing an application in 2027?"

# Invoke LLM
response = openai_client.responses.create(
    model="gpt-5.5",
    instructions=SYSTEM_PROMPT,
    input=WEB_USER_PROMPT,
    tools=[{"type": "web_search"}],
    include=[
        "web_search_call.action.sources",
        "web_search_call.results"
    ]
)

# Get text response
#response_text = response.output[0].content[0].text
response_text = response.output_text

# Show
print("***** Herramienta: web search *****")
print("Modelo Response:\n")
print(response_text)

# Get metadata response
metadata_json = get_response_metadata_openai(response)

# Show metadata
print("\n***** Metadata *****")
print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

# Get cost response
cost_json = get_calculate_cost_openai(response)

# Show cost response
print("\n***** Cost *****")
print(json.dumps(cost_json, indent=2, ensure_ascii=False))

***** Herramienta: web search *****
Modelo Response:

As of **May 27, 2026**, 2027 is still far enough out that regulations, AI tooling, and cloud economics may shift. But if I were modernizing an application for a 2027 operating model, I’d focus less on “move it to cloud” and more on **making the application evolvable, secure, observable, AI-ready, and economically sustainable**.

## 1. Start with business outcomes, not technology

Before choosing Kubernetes, serverless, microservices, AI agents, or a new framework, define:

- What business capability is this application supposed to improve?
- What must get faster: release cycles, onboarding, customer experience, compliance, scaling, cost control?
- What can be retired instead of modernized?
- What is the cost of doing nothing?
- Which parts of the system are actually constraining change?

Use a portfolio disposition model:

| Option | When to use |
|---|---|
| **Retire** | No longer used or replaced by another capability |
| **Retain

In [ ]:
# Extract the URLs the model consulted
print("=== Sources consulted ===")
for item in response.output:
    if item.type == "message":
        for block in item.content:
            if hasattr(block, "annotations") and block.annotations:
                for annotation in block.annotations:
                    if hasattr(annotation, "url"):
                        print(f"  - {annotation.url}")

=== Sources consulted ===
  - https://www.cncf.io/reports/the-cncf-annual-cloud-native-survey/
  - https://www.nist.gov/itl/ai-risk-management-framework
  - https://owasp.org/www-project-top-10-for-large-language-model-applications/
  - https://csrc.nist.gov/pubs/ir/8547/ipd
  - https://digital-strategy.ec.europa.eu/en/policies/regulatory-framework-ai
  - https://data.finops.org/


# Escalado de Costes

## Consideración 1: Coste en conversaciones multi-turno

### Aplicación de una conversación multi-turno

In [9]:
# Define system prompt: Role
SYSTEM_PROMPT = """
Acts as a senior software architect with experience in application modernization

You are helping a team to devise a strategy for modernising an application.
Ask clarifying questions where necessary.
Keep each answer to a maximum of three sentences.
"""

# Configure conversation history
conversation_history = []

# Configure prepared questions
questions = [
    "What are the main business goals driving this application modernization initiative?",
    "Can you help us assess the current state of the application, including architecture, technologies, dependencies, and pain points?",
    "Which parts of the application should we prioritize based on business criticality, technical risk, and modernization value?",
    "What modernization strategy would you recommend for this application: rehost, replatform, refactor, rearchitect, rebuild, or replace?",
    "What should the initial modernization roadmap look like, including phases, risks, quick wins, required team skills, and success metrics?",
]

print("***** ESCALADO - Consideración 1: Coste en conversaciones multi-turno *****\n")

# Print table header
print(
    f"{'Turn':>6s} "
    f"{'Input tok':>12s} "
    f"{'Output tok':>12s} "
    f"{'Cumulative cost':>18s}   "
    f"{'Question':>6s}"
)

print("-" * 100)

cumulative_cost = 0

for i, question in enumerate(questions):

    ## Add Question
    conversation_history.append({"role": "user", "content": question})

    # Invoke LLM
    response = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions=SYSTEM_PROMPT,
        input=conversation_history,
        max_output_tokens=300,
        temperature=0.3
    )

    # Add Response
    conversation_history.append({"role": "assistant", "content": response.output_text})

    # Get cost response
    cost_json = get_calculate_cost_openai(response)

    if cost_json:
        cumulative_cost += cost_json['cost']["total"]
        cost_str = f"${cumulative_cost:.6f}"

    # Print row
    print(
        f"{i+1:>6d} "
        f"{response.usage.input_tokens:>12d} "
        f"{response.usage.output_tokens:>12d} "
        f"{cost_str:>18s}   "
        f"{question[:40]:>40s}"
    )


***** ESCALADO - Consideración 1: Coste en conversaciones multi-turno *****

  Turn    Input tok   Output tok    Cumulative cost   Question
----------------------------------------------------------------------------------------------------
     1           68           50          $0.000040   What are the main business goals driving
     2          148           56          $0.000096   Can you help us assess the current state
     3          232           65          $0.000170   Which parts of the application should we
     4          331           56          $0.000253   What modernization strategy would you re
     5          419           68          $0.000357   What should the initial modernization ro


### Sobrecoste del systen prompt



In [ ]:
# Install SDK "OpenAI"
!pip install tiktoken

# Verify installation
import tiktoken

print(f"✅ tiktoken {tiktoken.__version__}")

✅ tiktoken 0.12.0


In [ ]:
# Configure model
MODEL = "gpt-4o-mini"

print(f"✅ Model selected: {MODEL}")

# Get the tokeniser corresponding to a specific model in the OpenAI API
tiktoken_client = tiktoken.encoding_for_model(MODEL)

print("✅ tiktoken configured successfully")

✅ Model selected: gpt-4o-mini
✅ tiktoken configured successfully


In [ ]:
# Configure system prompt
SYSTEM_PROMPT_USED=SYSTEM_PROMPT
#SYSTEM_PROMPT_USED=PRODUCTION_SYSTEM_PROMPT

# Configure num turns
num_turns = 20


# Pricing and Conditions
# - Estimated pricing for gpt-4.1-mini (USD per 1M tokens)
# - Cached tokens are not included
# - Adjust these values ​​according to the current official pricing
INPUT_PRICE_PER_MILLION = 0.15

print("***** ESCALADO - Consideración 1: Sobrecoste del System Prompt *****\n")

print("***** Detalles del System Prompt *****")
print(f"- System prompt utilizado:")
print(f"[{SYSTEM_PROMPT_USED.strip()}]")

# Get system prompts tokens (input type)
system_prompt_tokens = len(tiktoken_client.encode(SYSTEM_PROMPT_USED))
print(f"- Num tokens del system prompt utilizado: ~{system_prompt_tokens} tokens")

# Get system prompt cost
system_prompt_token_input_cost = (system_prompt_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
print(f"- Coste System prompt:  ${system_prompt_token_input_cost:.7f}")

print(f"\n***** Detalles del System Prompt en {num_turns} turnos *****")

print(f"IMPORTANTE: En cada turno se reenvía todo el historial de la conversación como tokens de entrada")


# Visualize the cost of the system prompt across turns
print(f"\nIncremento de tokens de entrada por turno:")

# Get system prompts tokens (input type) per all turns
num_turns_system_prompt_tokens = system_prompt_tokens * num_turns
print(f"- Num tokens del system prompt utilizado en todos los turnos: {system_prompt_tokens} x {num_turns} = ~{num_turns_system_prompt_tokens} tokens")

# Get system prompt cost per all turns
total_system_prompt_token_input_cost = (num_turns_system_prompt_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
print(f"- Coste System prompt en todos los turnos:  ${total_system_prompt_token_input_cost:.7f}")







***** ESCALADO - Consideración 1: Sobrecoste del System Prompt *****

***** Detalles del System Prompt *****
- System prompt utilizado:
[Act as a senior software architect with 15+ years of experience in application modernization.

# Objective
Your goal is to help the user analyze, design, and propose practical technical solutions to modernize legacy or existing applications, always considering software engineering best practices.

# Tone
- Didactic and professional
- Clear and structured explanations
- Practical and action-oriented
- Technically rigorous, but easy for experienced backend developers to follow

# Audience
- Backend developers with 5+ years of experience
- Technically proficient professionals familiar with software architecture, APIs, databases, deployment, and operational concerns

# Output Format
- Write in prose
- Do not use tables, emojis, bullet points o numbered list
- Maximum of 300 words
- Concise and actionable content
- Avoid generic statements; provide concret

## Consideración 2: Coste de escalado real

La cantidad de tokenes utilizados NO son muy grandes para la realidad actual

In [ ]:
# Configure general parameters
USERS = 1_000
DAYS_PER_MONTH = 22

# Configure calls parameters
CALLS_PER_USER_PER_DAY = 8
CALLS_PER_DAY = USERS * CALLS_PER_USER_PER_DAY
CALLS_PER_MONTH = CALLS_PER_DAY * DAYS_PER_MONTH

# Configure the average number of tokens used per cal
AVG_INPUT_TOKENS = 30_000
AVG_OUTPUT_TOKENS = 1_500

# Calculate monthly token estimation
INPUT_TOKENS_PER_MONTH = CALLS_PER_MONTH * AVG_INPUT_TOKENS
OUTPUT_TOKENS_PER_MONTH = CALLS_PER_MONTH * AVG_OUTPUT_TOKENS
TOTAL_TOKENS_PER_MONTH = INPUT_TOKENS_PER_MONTH + OUTPUT_TOKENS_PER_MONTH

# Configure available models
# - Prices are expressed in USD per 1M tokens
models_to_compare = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-5.4-mini": {"input": 0.75, "output": 4.50},
    "gpt-5.4": {"input": 2.50, "output": 15.00},
    "gpt-5.5-2026-04-23": {"input": 5.00, "output": 30.00}
}

print("***** ESCALADO - Consideración 2: Coste de escalado real *****\n")

print("Objetivo:")
print("Este ejemplo estima el coste mensual de uso de varios modelos LLM")
print("en un escenario donde múltiples usuarios realizan llamadas diarias a la API.\n")

print("***** Detalles del escenario *****")
print(f"- Número de usuarios: {USERS:,} usuarios")
print(f"- Días laborables estimados al mes: {DAYS_PER_MONTH} días")
print(f"- Llamadas por usuario y día: {CALLS_PER_USER_PER_DAY} llamadas/día")
print(f"- Total de llamadas por día: {CALLS_PER_DAY:,} llamadas/día")
print(f"- Total de llamadas por mes: {CALLS_PER_MONTH:,} llamadas/mes")

print("\n***** Tokens estimados por llamada *****")
print(f"- Tokens de entrada por llamada: {AVG_INPUT_TOKENS:,} input tokens")
print(f"- Tokens de salida por llamada: {AVG_OUTPUT_TOKENS:,} output tokens")

print("\n***** Tokens estimados por mes*****")
print(f"- Tokens de entrada al mes: {INPUT_TOKENS_PER_MONTH:,} input tokens")
print(f"- Tokens de salida al mes: {OUTPUT_TOKENS_PER_MONTH:,} output tokens")
print(f"- Tokens totales al mes: {TOTAL_TOKENS_PER_MONTH:,} tokens")

print("\n***** Fórmula de cálculo *****")
print("Coste por llamada =")
print("  (input_tokens / 1,000,000 * precio_input) +")
print("  (output_tokens / 1,000,000 * precio_output)")
print("\nCoste mensual = coste por llamada * total de llamadas al mes")

print("\n**** Calculo de costes de escalado *****")

# Create table header
print(
    f"\n{'Model':25s} "
    f"{'Cost/call':>15s} "
    f"{'Cost/day':>15s} "
    f"{'Cost/month':>15s} "
    f"{'Cost/user/month':>18s}"
)
print("-" * 95)

for model_name, prices in models_to_compare.items():

    input_cost_per_call = (AVG_INPUT_TOKENS / 1_000_000) * prices["input"]
    output_cost_per_call = (AVG_OUTPUT_TOKENS / 1_000_000) * prices["output"]
    cost_per_call = input_cost_per_call + output_cost_per_call

    daily_cost = cost_per_call * CALLS_PER_DAY
    monthly_cost = cost_per_call * CALLS_PER_MONTH
    monthly_cost_per_user = monthly_cost / USERS

    print(
        f"{model_name:25s} "
        f"{money(cost_per_call, 6):>15s} "
        f"{money(daily_cost, 2):>15s} "
        f"{money(monthly_cost, 2):>15s} "
        f"{money(monthly_cost_per_user, 2):>18s}"
    )

print("\n***** Lectura rápida *****")
print("- Cost/call indica cuánto cuesta una llamada media al modelo")
print("- Cost/day indica el coste diario para todos los usuarios")
print("- Cost/month indica el coste mensual total estimado")
print("- Cost/user/month indica el coste mensual medio por usuario")

***** ESCALADO - Consideración 2: Ejemplo de escalado real *****

Objetivo:
Este ejemplo estima el coste mensual de uso de varios modelos LLM
en un escenario donde múltiples usuarios realizan llamadas diarias a la API.

***** Detalles del escenario *****
- Número de usuarios: 1,000 usuarios
- Días laborables estimados al mes: 22 días
- Llamadas por usuario y día: 8 llamadas/día
- Total de llamadas por día: 8,000 llamadas/día
- Total de llamadas por mes: 176,000 llamadas/mes

***** Tokens estimados por llamada *****
- Tokens de entrada por llamada: 30,000 input tokens
- Tokens de salida por llamada: 1,500 output tokens

***** Tokens estimados por mes*****
- Tokens de entrada al mes: 5,280,000,000 input tokens
- Tokens de salida al mes: 264,000,000 output tokens
- Tokens totales al mes: 5,544,000,000 tokens

***** Fórmula de cálculo *****
Coste por llamada =
  (input_tokens / 1,000,000 * precio_input) +
  (output_tokens / 1,000,000 * precio_output)

Coste mensual = coste por llamada * to

## Consideración 3: Crecimiento del contexto en conversaciones multi-turno

La cantidad de tokenes utilizados NO son muy grandes para la realidad actual

In [ ]:
# Configure scenario
USERS = 1_000
CONVERSATIONS_PER_USER_PER_MONTH = 20
TURNS_PER_CONVERSATION = 12

# Token assumptions
SYSTEM_PROMPT_TOKENS = 350
AVG_USER_MESSAGE_TOKENS = 80
AVG_ASSISTANT_RESPONSE_TOKENS = 200

# Model pricing in USD per 1M tokens
MODEL_NAME = "gpt-4o-mini"
INPUT_PRICE_PER_1M = 0.15
OUTPUT_PRICE_PER_1M = 0.60

TOTAL_CONVERSATIONS_PER_MONTH = USERS * CONVERSATIONS_PER_USER_PER_MONTH

print("***** ESCALADO - Consideración 3: Escalado de coste en conversaciones multi-turno *****\n")

print("***** Detalles del escenario *****")
print(f"- Usuarios: {USERS:,}")
print(f"- Conversaciones por usuario al mes: {CONVERSATIONS_PER_USER_PER_MONTH}")
print(f"- Turnos por conversación: {TURNS_PER_CONVERSATION}")
print(f"- Conversaciones totales al mes: {TOTAL_CONVERSATIONS_PER_MONTH:,}")
print(f"- Modelo seleccionado: {MODEL_NAME}")

print("\n***** Supuestos de tokens *****")
print(f"- System prompt: {SYSTEM_PROMPT_TOKENS:,} tokens")
print(f"- Mensaje medio del usuario: {AVG_USER_MESSAGE_TOKENS:,} tokens")
print(f"- Respuesta media del asistente: {AVG_ASSISTANT_RESPONSE_TOKENS:,} tokens")

print("\n***** Simulación de una conversación *****")
print(
    f"\n{'Turn':>6s} "
    f"{'Input tokens':>15s} "
    f"{'Output tokens':>15s} "
    f"{'Cost/turn':>15s}"
)
print("-" * 60)

conversation_input_tokens = 0
conversation_output_tokens = 0
conversation_cost = 0

for turn in range(1, TURNS_PER_CONVERSATION + 1):

    # In each turn we resend:
    # - system prompt
    # - all previous user messages
    # - all previous assistant responses
    # - current user message
    input_tokens = (
        SYSTEM_PROMPT_TOKENS +
        turn * AVG_USER_MESSAGE_TOKENS +
        (turn - 1) * AVG_ASSISTANT_RESPONSE_TOKENS
    )

    output_tokens = AVG_ASSISTANT_RESPONSE_TOKENS

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_1M
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_1M
    turn_cost = input_cost + output_cost

    conversation_input_tokens += input_tokens
    conversation_output_tokens += output_tokens
    conversation_cost += turn_cost

    print(
        f"{turn:>6d} "
        f"{input_tokens:>15,d} "
        f"{output_tokens:>15,d} "
        f"{money(turn_cost, 8):>18s}"
    )

monthly_cost = conversation_cost * TOTAL_CONVERSATIONS_PER_MONTH

print("\n***** Resumen por conversación *****")
print(f"- Input tokens por conversación: {conversation_input_tokens:,}")
print(f"- Output tokens por conversación: {conversation_output_tokens:,}")
print(f"- Coste por conversación: ${conversation_cost:.6f}")

print("\n***** Estimación mensual *****")
print(f"- Conversaciones al mes: {TOTAL_CONVERSATIONS_PER_MONTH:,}")
print(f"- Coste mensual estimado: ${monthly_cost:,.2f}")

print("\n***** Lectura rápida *****")
print("- El coste por turno aumenta porque cada nueva llamada incluye más historial")
print("- Las conversaciones largas pueden ser mucho más caras que llamadas aisladas")
print("- Una estrategia de resumen de contexto puede reducir costes en escenarios multi-turno")

***** ESCALADO - Consideración 3: Crecimiento del contexto en conversaciones multi-turno *****

***** Detalles del escenario *****
- Usuarios: 1,000
- Conversaciones por usuario al mes: 20
- Turnos por conversación: 12
- Conversaciones totales al mes: 20,000
- Modelo seleccionado: gpt-4o-mini

***** Supuestos de tokens *****
- System prompt: 350 tokens
- Mensaje medio del usuario: 80 tokens
- Respuesta media del asistente: 200 tokens

***** Simulación de una conversación *****

  Turn    Input tokens   Output tokens       Cost/turn
------------------------------------------------------------
     1             430             200        $0.00018450
     2             710             200        $0.00022650
     3             990             200        $0.00026850
     4           1,270             200        $0.00031050
     5           1,550             200        $0.00035250
     6           1,830             200        $0.00039450
     7           2,110             200        $0.0004